# 03 — Silver Layer Transformation
## Customer Analytics Platform

Purpose: Clean, validate and enrich Bronze Delta tables.
This layer is the foundation for RFM feature engineering
and the CLV prediction model.

In [0]:
# load bronze delta tables for transformation

bronze_path = "/Volumes/workspace/default/olist_raw_data/bronze"
silver_path = "/Volumes/workspace/default/olist_raw_data/silver"

customers_df   = spark.read.format("delta").load(f"{bronze_path}/customers")
orders_df      = spark.read.format("delta").load(f"{bronze_path}/orders")
order_items_df = spark.read.format("delta").load(f"{bronze_path}/order_items")
payments_df    = spark.read.format("delta").load(f"{bronze_path}/payments")
reviews_df     = spark.read.format("delta").load(f"{bronze_path}/reviews")
products_df    = spark.read.format("delta").load(f"{bronze_path}/products")
sellers_df     = spark.read.format("delta").load(f"{bronze_path}/sellers")

print("Bronze tables loaded for silver transformation")

In [0]:
# filter only delivered orders and drop nulls in key columns

from pyspark.sql.functions import col

# keep only delivered orders — undelivered dont count for CLV
orders_clean = orders_df \
    .filter(col("order_status") == "delivered") \
    .dropna(subset=["order_id", "customer_id", 
                    "order_purchase_timestamp",
                    "order_delivered_customer_date"])

print(f"orders before cleaning : {orders_df.count()}")
print(f"orders after cleaning  : {orders_clean.count()}")
print(f"rows removed           : {orders_df.count() - orders_clean.count()}")

In [0]:
# clean customers table — drop duplicates on customer_unique_id

customers_clean = customers_df \
    .dropna(subset=["customer_id", "customer_unique_id"]) \
    .dropDuplicates(["customer_unique_id"])

print(f"customers before cleaning : {customers_df.count()}")
print(f"customers after cleaning  : {customers_clean.count()}")

In [0]:
# clean order_items — drop nulls and add total item value column

from pyspark.sql.functions import round as spark_round

order_items_clean = order_items_df \
    .dropna(subset=["order_id", "product_id", "price"]) \
    .dropDuplicates(["order_id", "order_item_id"]) \
    .withColumn("item_total_value", 
                spark_round(col("price") + col("freight_value"), 2))

print(f"order_items before cleaning : {order_items_df.count()}")
print(f"order_items after cleaning  : {order_items_clean.count()}")

In [0]:
# clean payments — keep one payment summary per order

from pyspark.sql.functions import sum as spark_sum, round as spark_round

payments_clean = payments_df \
    .groupBy("order_id") \
    .agg(
        spark_round(spark_sum("payment_value"), 2)
        .alias("total_payment_value"),
        spark_sum("payment_installments")
        .alias("total_installments")
    )

print(f"payments before aggregation : {payments_df.count()}")
print(f"payments after aggregation  : {payments_clean.count()}")
print("\nsample output:")
payments_clean.show(5)

In [0]:
# clean reviews — keep only review score and comment, drop heavy nulls

from pyspark.sql.functions import when, lit

reviews_clean = reviews_df \
    .dropna(subset=["order_id", "review_score"]) \
    .dropDuplicates(["order_id"]) \
    .withColumn("has_comment",
                when(col("review_comment_message").isNull(), 0)
                .otherwise(1)) \
    .select("order_id", 
            "review_score", 
            "review_comment_message",
            "has_comment")

print(f"reviews before cleaning : {reviews_df.count()}")
print(f"reviews after cleaning  : {reviews_clean.count()}")
print("\nreview score distribution:")
reviews_clean.groupBy("review_score") \
    .count() \
    .orderBy("review_score") \
    .show()

In [0]:
# reload reviews from bronze and properly extract numeric score only

from pyspark.sql.functions import regexp_extract, col

reviews_fixed = reviews_df \
    .dropDuplicates(["order_id"]) \
    .withColumn("review_score_clean",
                regexp_extract(col("review_score")
                .cast("string"), r"^([1-5])$", 1)) \
    .filter(col("review_score_clean") != "") \
    .withColumn("review_score", 
                col("review_score_clean").cast("integer")) \
    .withColumn("has_comment",
                when(col("review_comment_message").isNull(), 0)
                .otherwise(1)) \
    .select("order_id",
            "review_score",
            "review_comment_message",
            "has_comment")

# overwrite reviews_clean with fixed version
reviews_clean = reviews_fixed

print(f"reviews after fix : {reviews_clean.count()}")
print("\nreview score distribution:")
reviews_clean.groupBy("review_score") \
    .count() \
    .orderBy("review_score") \
    .show()

In [0]:
# add delivery days column to orders — key feature for CLV model

from pyspark.sql.functions import datediff, col, round as spark_round

orders_enriched = orders_clean \
    .withColumn("delivery_days",
                datediff(
                    col("order_delivered_customer_date"),
                    col("order_purchase_timestamp")
                )) \
    .withColumn("is_late_delivery",
                when(
                    col("order_delivered_customer_date") > 
                    col("order_estimated_delivery_date"), 1)
                .otherwise(0))

print(f"orders enriched : {orders_enriched.count()}")
print("\ndelivery days stats:")
orders_enriched.select(
    "delivery_days",
    "is_late_delivery"
).summary("mean", "min", "max").show()

In [0]:
# save all cleaned tables as silver delta tables

print("Saving Silver Delta tables...")
print("-" * 40)

customers_clean.write.format("delta") \
    .mode("overwrite").save(f"{silver_path}/customers")
print("customers saved")

orders_enriched.write.format("delta") \
    .mode("overwrite").save(f"{silver_path}/orders")
print("orders saved")

order_items_clean.write.format("delta") \
    .mode("overwrite").save(f"{silver_path}/order_items")
print("order_items saved")

payments_clean.write.format("delta") \
    .mode("overwrite").save(f"{silver_path}/payments")
print("payments saved")

reviews_clean.write.format("delta") \
    .mode("overwrite").save(f"{silver_path}/reviews")
print("reviews saved")

products_df.write.format("delta") \
    .mode("overwrite").save(f"{silver_path}/products")
print("products saved")

sellers_df.write.format("delta") \
    .mode("overwrite").save(f"{silver_path}/sellers")
print("sellers saved")

print("-" * 40)
print("All Silver Delta tables saved successfully!")

In [0]:
# verify silver delta tables row counts

silver_tables = {
    "customers"   : f"{silver_path}/customers",
    "orders"      : f"{silver_path}/orders",
    "order_items" : f"{silver_path}/order_items",
    "payments"    : f"{silver_path}/payments",
    "reviews"     : f"{silver_path}/reviews",
    "products"    : f"{silver_path}/products",
    "sellers"     : f"{silver_path}/sellers"
}

print(f"{'Table':<15} {'Row Count':>10} {'Status':>10}")
print("-" * 40)

for table_name, path in silver_tables.items():
    df = spark.read.format("delta").load(path)
    print(f"{table_name:<15} {df.count():>10} {'OK':>10}")

print("-" * 40)
print("Silver verification done")